# 🚀 Notebook 03 — Vertex AI Deployment
**FMCG Promotional Analytics | Portfolio Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Panosemmanouilidis2/fmcg-ds-technical-portfolio/blob/main/promotional-analytics/notebooks/03_model_training.ipynb)

This notebook deploys the trained XGBoost model to a live Google Cloud Vertex AI endpoint.

---

## ✅ Before You Start — Checklist

| Step | Action |
|---|---|
| 1 | Run `02_feature_engineering.ipynb` — confirms `models/xgb_tuned.json` exists |
| 2 | Create a GCP project at [console.cloud.google.com](https://console.cloud.google.com) |
| 3 | Enable **Vertex AI API** and **Cloud Storage API** in your project |
| 4 | Run `gcloud auth application-default login` in your terminal |
| 5 | Fill in **Cell 0** below — only block you need to edit |

**Run time after setup:** ~15 minutes (endpoint provisioning)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 0 — YOUR GCP CONFIGURATION  ← only block you need to edit
# ══════════════════════════════════════════════════════════════════

PROJECT_ID  = "YOUR_PROJECT_ID"     # GCP Console → top of any page
REGION      = "europe-west2"        # change only if your project is in a different region
BUCKET_NAME = "YOUR_BUCKET_NAME"    # new or existing GCS bucket (no gs:// prefix)
MODEL_NAME  = "fmcg-xgb-promo-v1"  # display name shown in Vertex AI Model Registry

# ── Derived — do not edit below this line ─────────────────────────
BUCKET_URI        = f"gs://{BUCKET_NAME}"
GCS_MODEL_DIR     = f"{BUCKET_URI}/artifacts/model"
SKLEARN_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"

print("=== CONFIG ===")
print(f"  Project  : {PROJECT_ID}")
print(f"  Region   : {REGION}")
print(f"  Bucket   : {BUCKET_URI}")
print(f"  Model    : {MODEL_NAME}")
print()
if "YOUR_" in PROJECT_ID or "YOUR_" in BUCKET_NAME:
    print("⚠️  Please fill in PROJECT_ID and BUCKET_NAME above before continuing")
else:
    print("✅ Config looks good — proceed to Cell 1")


## Step 1 — Install & Authenticate

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 1 — INSTALL LIBRARIES
# ══════════════════════════════════════════════════════════════════
!pip install google-cloud-aiplatform google-cloud-storage xgboost==1.7.6 --quiet
print("✅ Libraries installed")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — AUTHENTICATE
# On Vertex AI Workbench: skip this cell — credentials are automatic.
# On Colab or local: run  gcloud auth application-default login
# in your terminal first, then execute this cell.
# ══════════════════════════════════════════════════════════════════
import google.auth
import google.auth.transport.requests

credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())
print(f"✅ Authenticated — detected project: {detected_project}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — IMPORTS & INITIALISE VERTEX AI SDK
# ══════════════════════════════════════════════════════════════════
import os, json, pickle, shutil
import numpy as np
import xgboost as xgb
from google.cloud import aiplatform, storage

aiplatform.init(project=PROJECT_ID, location=REGION)
print(f"✅ Vertex AI SDK initialised — {PROJECT_ID} / {REGION}")


## Step 2 — Create GCS Bucket & Upload Model Artefacts

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — CREATE BUCKET  (skips safely if it already exists)
# ══════════════════════════════════════════════════════════════════
client = storage.Client(project=PROJECT_ID)

try:
    bucket = client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"✅ Bucket created: {BUCKET_URI}")
except Exception as e:
    if "already own" in str(e).lower() or "already exists" in str(e).lower():
        bucket = client.bucket(BUCKET_NAME)
        print(f"✅ Bucket already exists: {BUCKET_URI}  (no action needed)")
    else:
        raise


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — STAGE MODEL ARTEFACTS LOCALLY
# Vertex AI reads from a GCS folder — we copy files into staging/
# before uploading. Model must be saved at XGBoost 1.7.6 for
# compatibility with the Vertex AI sklearn serving container.
# ══════════════════════════════════════════════════════════════════
os.makedirs('staging/model', exist_ok=True)

booster = xgb.Booster()
booster.load_model('models/xgb_tuned.json')
booster.save_model('staging/model/model.json')
print("✅ Model staged at staging/model/model.json")

for f in ['feature_cols.json', 'feature_medians.json']:
    shutil.copy(f'models/{f}', f'staging/model/{f}')
    print(f"✅ Staged models/{f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — UPLOAD ARTEFACTS TO GCS
# ══════════════════════════════════════════════════════════════════
for fname in os.listdir('staging/model'):
    local_path = f'staging/model/{fname}'
    blob_name  = f'artifacts/model/{fname}'
    bucket.blob(blob_name).upload_from_filename(local_path)
    print(f"  Uploaded → gs://{BUCKET_NAME}/{blob_name}")

print(f"\n✅ All artefacts uploaded to {GCS_MODEL_DIR}")


## Step 3 — Register Model in Vertex AI Model Registry

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — REGISTER MODEL
# ⏳ Takes ~3–5 minutes
# ══════════════════════════════════════════════════════════════════
print(f"Registering '{MODEL_NAME}'... (3–5 min)")

model = aiplatform.Model.upload(
    display_name                = MODEL_NAME,
    artifact_uri                = GCS_MODEL_DIR,
    serving_container_image_uri = SKLEARN_CONTAINER,
    description                 = "XGBoost promo sell-out forecaster — Market A & B",
)

print(f"\n✅ Model registered")
print(f"   Resource : {model.resource_name}")


## Step 4 — Create Endpoint & Deploy

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — CREATE ENDPOINT
# ⏳ Takes ~2–3 minutes
# ══════════════════════════════════════════════════════════════════
print("Creating endpoint...")

endpoint = aiplatform.Endpoint.create(
    display_name = f"{MODEL_NAME}-endpoint",
    description  = "FMCG promotional sell-out forecaster"
)

# Save resource name for future sessions
with open('models/endpoint_resource_name.txt', 'w') as f:
    f.write(endpoint.resource_name)

print(f"\n✅ Endpoint created")
print(f"   Resource : {endpoint.resource_name}")
print(f"   Saved    → models/endpoint_resource_name.txt")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — DEPLOY MODEL TO ENDPOINT
# ⏳ Takes 10–15 minutes — do NOT close the notebook
# ══════════════════════════════════════════════════════════════════
print(f"Deploying {MODEL_NAME}...")
print("⏳ This takes 10–15 minutes — please wait\n")

endpoint.deploy(
    model                       = model,
    deployed_model_display_name = f"{MODEL_NAME}-deployed",
    machine_type                = "n1-standard-2",
    min_replica_count           = 1,
    max_replica_count           = 1,
    traffic_percentage          = 100,
)

print("\n✅ Model deployed — endpoint is ACTIVE")


## Step 5 — Smoke Test

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — SMOKE TEST: one live prediction from the endpoint
# Builds a payload from median feature values and calls the endpoint.
# Expected result: a positive integer (predicted sell-out units).
# ══════════════════════════════════════════════════════════════════
import json, numpy as np

feature_cols = json.load(open('models/feature_cols.json'))
medians      = json.load(open('models/feature_medians.json'))

payload    = [float(medians.get(col, 0.0)) for col in feature_cols]
response   = endpoint.predict(instances=[payload])
raw_pred   = response.predictions[0]
pred_units = max(0, round(np.expm1(raw_pred)))

print("=== SMOKE TEST RESULT ===")
print(f"  Payload length  : {len(payload)}")
print(f"  Raw log pred    : {raw_pred:.4f}")
print(f"  Predicted units : {pred_units:,}")
print()
if pred_units > 0:
    print("✅ Endpoint is live and returning valid predictions")
else:
    print("⚠️  Prediction returned 0 — check feature alignment in Cell 5")


## Step 6 — Cost Control

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — UNDEPLOY  (stops billing — run at end of each session)
# The endpoint and model registration are preserved.
# To restore: re-run Cells 8–10 only (~15 min).
# ══════════════════════════════════════════════════════════════════

# ⚠️  Uncomment the lines below when you want to stop the endpoint:

# for deployed_model in endpoint.list_models():
#     print(f"Undeploying: {deployed_model.display_name}...")
#     endpoint.undeploy(deployed_model_id=deployed_model.id)
#     print("  ✅ Done")
# print("\nEndpoint stopped — billing halted")
# print("To redeploy: re-run Cells 8–10 only")

print("ℹ️  Undeploy block is commented out — uncomment lines above to stop billing")
